# Manual de bases de datos vectoriales
## Fase 1 — La base matemática: vectores, numpy, matplotlib y distancias

**Objetivo de esta fase:** entender, desde cero, qué es un vector y cómo se mide qué tan "parecidos" o qué tan "lejanos" son dos vectores. Todo lo que viene después (embeddings, búsqueda semántica, pgvector, RAG) se apoya en estas ideas.

**Qué vas a aprender**

| # | Tema |
|---|------|
| 0 | Preparar el entorno (Python 3.12, entorno virtual, VS Code) |
| 1 | Qué es un vector |
| 2 | numpy desde cero |
| 3 | matplotlib desde cero |
| 4 | Operaciones con vectores (suma, resta, escalar) |
| 5 | Producto punto |
| 6 | Norma (la "longitud" de un vector) |
| 7 | Distancia euclidiana |
| 8 | Similitud coseno |
| 9 | Dos funciones reutilizables |
| 10 | Cuándo usar una métrica y cuándo la otra |
| 11 | Vectores de muchas dimensiones |
| 12 | Resumen |

### Cómo usar este manual

- Es un **manual**, no una serie de ejercicios: todo el código viene completo y funcional. La idea es que lo leas, lo ejecutes, lo cambies para ver qué pasa, y copies a tu archivo limpio lo que quieras conservar.
- **Las celdas dependen de las anteriores** (variables y funciones que se definen antes). Si copias el código a un archivo `.py`, cópialo en orden.
- Todas las gráficas terminan con `plt.show()`. En el notebook no es obligatorio, pero en un archivo `.py` **sí**, y así el código queda listo para copiar.
- En cada concepto seguiremos el mismo orden: **idea intuitiva → fórmula → cálculo a mano con números chicos → código → comprobación**.
- Los cálculos a mano están escritos en las celdas de texto; el código los **comprueba**. Si algún resultado no coincide, quiero saberlo.

> **Sobre lo que no está verificado:** los tiempos de ejecución que verás en la sección 2 y los tamaños de vector que menciono como "comunes" (384, 768, 1536...) son valores típicos que conozco de forma general, no medidos por mí en tu equipo. Los tiempos cambiarán en tu máquina.

---
# 0. Preparar el entorno

## ¿Por qué un entorno virtual?

Un **entorno virtual** (`venv`) es una carpeta con su propia copia de Python y sus propias librerías. Sirve para que las librerías de este proyecto (numpy, matplotlib, y más adelante otras) **no se mezclen** con las de otros proyectos ni con el Python global de tu equipo. Es el equivalente a tener un `vendor/` por proyecto en PHP/Composer.

## Paso 1 — Crear el proyecto y el entorno

Abre una terminal (PowerShell) en la carpeta donde quieras trabajar y ejecuta:

```powershell
mkdir vectores
cd vectores
py -3.12 -m venv .venv
```

- `py -3.12` usa el lanzador de Python de Windows para pedir específicamente la versión 3.12.
- `-m venv .venv` crea el entorno dentro de una carpeta oculta llamada `.venv`.

## Paso 2 — Activarlo

```powershell
.venv\Scripts\Activate.ps1
```

Si funciona, verás `(.venv)` al inicio de la línea de la terminal.

**Si PowerShell dice que la ejecución de scripts está deshabilitada:** es la política de ejecución de Windows. Tienes dos caminos: usar `cmd` y activar con `.venv\Scripts\activate.bat`, o cambiar la política de PowerShell. Lo segundo es un cambio de configuración de tu equipo, así que decídelo tú según tus políticas de TI.

## Paso 3 — Instalar las librerías

Con el entorno activo:

```powershell
python -m pip install --upgrade pip
pip install numpy matplotlib ipykernel
```

- `numpy`: cálculo con arreglos numéricos (lo vemos en la sección 2).
- `matplotlib`: gráficas (sección 3).
- `ipykernel`: permite que VS Code ejecute notebooks con este entorno.

## Paso 4 — Abrir este notebook en VS Code con ese entorno

1. Instala en VS Code las extensiones **Python** y **Jupyter** (ambas de Microsoft), si no las tienes.
2. Abre este archivo `.ipynb`.
3. Arriba a la derecha pulsa **Select Kernel** → **Python Environments** → elige el que dice `.venv`. (Los nombres exactos del menú pueden variar un poco según tu versión de VS Code.)
4. Ejecuta una celda con `Shift + Enter`.

Ejecuta la siguiente celda para confirmar que todo está bien.

In [ ]:
import sys
import numpy
import matplotlib

print("Python    :", sys.version.split()[0])
print("numpy     :", numpy.__version__)
print("matplotlib:", matplotlib.__version__)
print("Ejecutable:", sys.executable)

Revisa la última línea: `Ejecutable` debe apuntar dentro de tu carpeta `.venv`. Si apunta al Python global, el kernel seleccionado no es el correcto y las librerías que instalaste no se estarían usando.

---
# 1. Qué es un vector

## Idea intuitiva

Un **vector** es, en lo que nos interesa, **una lista ordenada de números**. Nada más.

La forma más fácil de verlo es con coordenadas en un mapa. El punto `(2, 3)` significa "2 pasos a la derecha y 3 pasos hacia arriba". Esos dos números son un vector de **2 dimensiones**.

Un vector se puede imaginar de dos maneras, y las dos son válidas:

1. **Como un punto** en el espacio: el lugar donde cae `(2, 3)`.
2. **Como una flecha** que sale del origen `(0, 0)` y termina en ese punto.

Usaremos las dos, según convenga.

## De 2 dimensiones a cientos

Nada impide que la lista tenga más números. Un vector `[0.12, -0.50, 0.33, ...]` con 384 números es un vector de **384 dimensiones**. No se puede dibujar, pero **todas las operaciones que aprenderemos funcionan igual** que con 2 números. Por eso aprendemos con 2: se pueden ver y comprobar con los ojos.

## ¿Qué tiene que ver con las bases de datos vectoriales?

Un **embedding** es un vector que un modelo de IA calcula a partir de un texto (o imagen, etc.). Cada número del vector captura algún aspecto del significado que el modelo aprendió, pero **un número por sí solo no tiene una interpretación humana** ("la dimensión 57 significa X" no es algo que se pueda decir). Lo que sí funciona es comparar vectores completos: textos con significado parecido producen vectores cercanos.

Toda la fase 1 consiste en aprender a medir esa cercanía.

## Vectores con listas de Python

Antes de numpy, veamos un vector con lo que Python ya trae: una lista.

In [ ]:
a = [2, 3]   # un vector de 2 dimensiones
b = [5, 1]   # otro vector de 2 dimensiones

print("a =", a)
print("b =", b)
print("Dimensiones de a:", len(a))    # len() cuenta cuántos números tiene
print("Primer número de a:", a[0])    # los índices empiezan en 0
print("Segundo número de a:", a[1])

Estos mismos dos vectores, `a = [2, 3]` y `b = [5, 1]`, los usaremos en casi todo el manual, para que los resultados sean fáciles de seguir y comprobar.

**El problema de las listas:** si intentas sumar dos listas, Python no suma número por número, las **concatena**. Lo vemos en la siguiente sección, y es justamente la razón por la que existe numpy.

---
# 2. numpy desde cero

## ¿Qué es numpy y por qué se usa?

**numpy** es la librería base de Python para trabajar con números en bloque. Su pieza central es el **array** (`numpy.ndarray`): una lista de números **del mismo tipo**, guardada de forma compacta en memoria, con operaciones matemáticas ya incorporadas.

Dos razones para usarlo en lugar de listas:

1. **Las operaciones son "elemento por elemento"**: `a + b` suma número con número, que es lo que significa sumar vectores.
2. **Es más rápido** con muchos datos, porque los cálculos los hace código compilado y no un ciclo de Python.

Por convención siempre se importa con el alias `np`.

In [ ]:
import numpy as np

# --- Listas de Python: "+" concatena ---
lista_a = [2, 3]
lista_b = [5, 1]
print("Listas   a + b =", lista_a + lista_b)

# --- Arrays de numpy: "+" suma elemento por elemento ---
a = np.array([2.0, 3.0])
b = np.array([5.0, 1.0])
print("Arrays   a + b =", a + b)

Con listas, `[2, 3] + [5, 1]` da `[2, 3, 5, 1]` (una lista más larga). Con arrays, da `[7., 4.]`: la suma vectorial que esperamos.

Fíjate que escribí `2.0` y `3.0` y no `2` y `3`: así los arrays son de números **decimales** (`float`), que es lo que usaremos siempre con embeddings.

## Anatomía de un array

Tres atributos que verás constantemente:

- `shape`: la forma, es decir, cuántos elementos hay en cada eje. Para un vector de 2 números es `(2,)`. El `,` es porque es una tupla de un solo elemento.
- `ndim`: cuántos ejes tiene. Un vector tiene 1.
- `dtype`: el tipo de dato de los números.

In [ ]:
a = np.array([2.0, 3.0])

print("Contenido :", a)
print("shape     :", a.shape)   # (2,)  -> 2 elementos en un solo eje
print("ndim      :", a.ndim)    # 1     -> es un vector (un solo eje)
print("dtype     :", a.dtype)   # float64 -> decimales de 64 bits
print("Primer elemento:", a[0]) # se indexa igual que una lista

## Operaciones elemento por elemento

Con arrays, los operadores matemáticos se aplican a **cada** elemento. Esto se llama operar "en bloque" (o *vectorizar*).

In [ ]:
a = np.array([2.0, 3.0])
b = np.array([5.0, 1.0])

print("a + b  =", a + b)      # [2+5, 3+1]
print("a - b  =", a - b)      # [2-5, 3-1]
print("a * b  =", a * b)      # [2*5, 3*1]  (¡ojo! esto NO es el producto punto, lo vemos en la sección 5)
print("a * 2  =", a * 2)      # [2*2, 3*2]  un número contra un array se aplica a todos
print("a ** 2 =", a ** 2)     # [2^2, 3^2]  cada elemento al cuadrado
print("sqrt(a)=", np.sqrt(a)) # raíz cuadrada de cada elemento
print("suma de a =", np.sum(a))  # suma todos los elementos: 2+3

Retén estas dos últimas: **`a ** 2`** (elevar al cuadrado cada elemento) y **`np.sum(...)`** (sumar todos los elementos). Con ellas construiremos la norma y la distancia.

## Comparación de velocidad (opcional)

Este experimento compara duplicar un millón de números con una lista (ciclo de Python) contra un array de numpy. **Los tiempos que verás dependen de tu equipo**, lo importante es la proporción.

In [ ]:
import time

n = 1_000_000   # el guion bajo es solo para leer mejor: 1_000_000 = 1000000
lista_grande = list(range(n))
array_grande = np.arange(n)   # equivalente a range(n), pero como array

inicio = time.perf_counter()
resultado_lista = [x * 2 for x in lista_grande]
t_lista = time.perf_counter() - inicio

inicio = time.perf_counter()
resultado_array = array_grande * 2
t_array = time.perf_counter() - inicio

print(f"Lista de Python: {t_lista:.4f} s")
print(f"Array de numpy : {t_array:.4f} s")
print(f"numpy fue aproximadamente {t_lista / t_array:.0f} veces más rápido en esta ejecución")

Con vectores de 2 números esto no importa. Con miles o millones de vectores de cientos de dimensiones (el caso real de una base vectorial), importa mucho.

---
# 3. matplotlib desde cero

## ¿Qué es matplotlib?

**matplotlib** es la librería estándar de Python para hacer gráficas. Se importa por convención así:

```python
import matplotlib.pyplot as plt
```

## Los dos conceptos que hay que entender

- **Figure (`fig`)**: la ventana o lienzo completo.
- **Axes (`ax`)**: un área de dibujo dentro del lienzo, con sus ejes X e Y. Una figura puede tener varios `ax`.

El patrón que usaremos siempre:

```python
fig, ax = plt.subplots(figsize=(6, 6))   # crea el lienzo y un área de dibujo (ancho, alto en pulgadas)
ax.<algo>(...)                           # dibujas sobre el área (puntos, líneas, textos, títulos)
plt.show()                               # muestra la figura
```

## Primera gráfica: dos puntos

`ax.scatter(x, y)` dibuja puntos. Recibe **todas las x juntas** y **todas las y juntas**, y no cada punto por separado.

In [ ]:
import matplotlib.pyplot as plt

a = np.array([2.0, 3.0])
b = np.array([5.0, 1.0])

fig, ax = plt.subplots(figsize=(5, 5))

# scatter(lista_de_x, lista_de_y)
ax.scatter([a[0], b[0]], [a[1], b[1]], s=80)   # s = tamaño del punto

ax.set_xlim(0, 7)        # rango visible del eje X
ax.set_ylim(0, 5)        # rango visible del eje Y
ax.set_title("Dos vectores vistos como puntos")
ax.set_xlabel("dimensión 1 (x)")
ax.set_ylabel("dimensión 2 (y)")
ax.grid(True)            # cuadrícula

plt.show()

## Dibujar vectores como flechas

matplotlib no tiene una función "dibuja un vector" lista para usar, así que vamos a armar **dos funciones auxiliares** que reutilizaremos en todo el manual.

- `preparar_plano(...)`: crea una figura con ejes de la misma escala (importante: si X e Y tienen escalas distintas, los ángulos se ven deformados), cuadrícula y los ejes en 0.
- `dibujar_flecha(...)`: dibuja una flecha desde un origen hasta el final del vector, con una etiqueta. Usa `ax.annotate`, que sirve para dibujar flechas cuando se le da texto vacío y `arrowprops`.

In [ ]:
def preparar_plano(titulo, limite_x=(-1, 8), limite_y=(-1, 8)):
    '''Crea una figura con un plano cartesiano listo para dibujar vectores.'''
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(*limite_x)          # *tupla desempaqueta: set_xlim(-1, 8)
    ax.set_ylim(*limite_y)
    ax.set_aspect("equal")          # misma escala en X e Y
    ax.grid(True, linestyle="--", alpha=0.4)   # alpha = transparencia
    ax.axhline(0, color="black", linewidth=0.8)  # línea horizontal en y = 0
    ax.axvline(0, color="black", linewidth=0.8)  # línea vertical en x = 0
    ax.set_title(titulo)
    ax.set_xlabel("dimensión 1 (x)")
    ax.set_ylabel("dimensión 2 (y)")
    return fig, ax


def dibujar_flecha(ax, vector, etiqueta, color, origen=(0, 0)):
    '''Dibuja un vector como flecha desde 'origen' hasta origen + vector.'''
    destino = (origen[0] + vector[0], origen[1] + vector[1])
    ax.annotate(
        "",                  # sin texto: solo queremos la flecha
        xy=destino,          # hacia dónde apunta la flecha
        xytext=origen,       # desde dónde sale
        arrowprops=dict(arrowstyle="->", color=color, linewidth=2),
    )
    ax.text(destino[0] + 0.15, destino[1] + 0.15, etiqueta, color=color, fontsize=12)


# Probamos nuestras funciones con a y b
fig, ax = preparar_plano("a = [2, 3] y b = [5, 1] como flechas")
dibujar_flecha(ax, a, "a", "tab:blue")
dibujar_flecha(ax, b, "b", "tab:red")
plt.show()

Ahora cada vector es una flecha que sale del origen. Dos cosas se pueden **ver** en esta gráfica y serán clave más adelante:

- La **longitud** de cada flecha (qué tan "grande" es el vector).
- El **ángulo** entre las flechas (qué tanto apuntan hacia el mismo lado).

---
# 4. Operaciones con vectores

Usaremos `a = [2, 3]` y `b = [5, 1]`.

## Suma

Se suma **componente con componente** (posición con posición):

$$a + b = [\,2+5,\; 3+1\,] = [\,7,\; 4\,]$$

**Interpretación geométrica:** si caminas la flecha `a` y desde su punta caminas la flecha `b`, llegas a la punta de `a + b`.

## Resta

$$a - b = [\,2-5,\; 3-1\,] = [\,-3,\; 2\,]$$

Esta es la importante para nosotros: **`a - b` es la flecha que va de la punta de `b` a la punta de `a`**. Por eso la resta aparecerá en la distancia (sección 7).

## Multiplicar por un número (escalar)

Se multiplica **cada** componente:

$$2 \cdot a = [\,2\cdot 2,\; 2\cdot 3\,] = [\,4,\; 6\,]$$

**Interpretación geométrica:** la flecha mantiene **la misma dirección** pero se estira al doble. Esto será importante en la sección 10.

In [ ]:
a = np.array([2.0, 3.0])
b = np.array([5.0, 1.0])

suma   = a + b
resta  = a - b
doble  = 2 * a

print("a + b  =", suma)    # esperado a mano: [7, 4]
print("a - b  =", resta)   # esperado a mano: [-3, 2]
print("2 * a  =", doble)   # esperado a mano: [4, 6]

Los tres resultados coinciden con los cálculos a mano. Ahora la suma **vista**: dibujamos `a`, `b` y `a + b`, y además una copia de `b` colocada en la punta de `a` (flecha punteada) para ver el "camina una y luego la otra".

In [ ]:
fig, ax = preparar_plano("Suma de vectores: a + b", limite_x=(-1, 9), limite_y=(-1, 6))

dibujar_flecha(ax, a, "a", "tab:blue")
dibujar_flecha(ax, b, "b", "tab:red")
dibujar_flecha(ax, suma, "a + b", "tab:green")

# b copiada con origen en la punta de a: llega exactamente a la punta de a + b
dibujar_flecha(ax, b, "", "salmon", origen=(a[0], a[1]))

plt.show()

---
# 5. Producto punto

## Idea intuitiva

El **producto punto** toma dos vectores y devuelve **un solo número** que dice **qué tanto apuntan en la misma dirección**:

- Número **positivo y grande** → apuntan hacia el mismo lado.
- **Cero** → son perpendiculares (forman 90°).
- Número **negativo** → apuntan hacia lados opuestos.

## Fórmula

Se multiplican las componentes de la misma posición y se **suman** los resultados:

$$a \cdot b = a_1 b_1 + a_2 b_2 + \dots + a_n b_n$$

## Cálculo a mano con `a = [2, 3]` y `b = [5, 1]`

$$a \cdot b = (2 \cdot 5) + (3 \cdot 1) = 10 + 3 = 13$$

Fíjate que el resultado es **un número**, no un vector. Es muy distinto de `a * b` en numpy, que da un vector `[10, 3]`: el producto punto es **esa multiplicación seguida de la suma** (`10 + 3`).

In [ ]:
a = np.array([2.0, 3.0])
b = np.array([5.0, 1.0])

# Forma 1: paso a paso, igual que a mano
multiplicados = a * b                 # [2*5, 3*1] = [10, 3]
punto_manual = np.sum(multiplicados)  # 10 + 3
print("a * b (elemento por elemento):", multiplicados)
print("Suma de eso (producto punto)  :", punto_manual)

# Forma 2: la función de numpy
print("np.dot(a, b)                  :", np.dot(a, b))

# Forma 3: el operador @ (hace lo mismo con vectores)
print("a @ b                         :", a @ b)

Las tres formas dan **13**, igual que el cálculo a mano.

## Los tres casos de signo, con ejemplos

Veamos las tres situaciones que describí al inicio con vectores muy simples:

In [ ]:
mismo_lado = np.array([1.0, 0.0])
perpendicular = np.array([0.0, 1.0])
lado_opuesto = np.array([-1.0, 0.0])

ref = np.array([1.0, 0.0])   # vector de referencia: apunta a la derecha

print("[1,0] · [1,0]  =", ref @ mismo_lado,    "-> mismo lado     (positivo)")
print("[1,0] · [0,1]  =", ref @ perpendicular, "-> perpendiculares (cero)")
print("[1,0] · [-1,0] =", ref @ lado_opuesto,  "-> lados opuestos  (negativo)")

## Su significado geométrico (lo usaremos en la sección 8)

Existe una segunda fórmula del producto punto, equivalente a la primera:

$$a \cdot b = \|a\| \cdot \|b\| \cdot \cos(\theta)$$

donde $\|a\|$ y $\|b\|$ son las longitudes (normas) de los vectores y $\theta$ es el ángulo entre ellos. De aquí saldrá la similitud coseno: si despejamos $\cos(\theta)$, obtenemos una medida del ángulo. Primero necesitamos entender qué es la norma.

---
# 6. Norma (la "longitud" de un vector)

## Idea intuitiva

La **norma** de un vector es **la longitud de su flecha**. Se escribe $\|a\|$.

## De dónde sale la fórmula: Pitágoras

Tomemos `v = [3, 4]`. Su flecha va 3 pasos a la derecha y 4 hacia arriba. Eso forma un triángulo rectángulo con catetos de 3 y 4, y la flecha es la hipotenusa. Por el teorema de Pitágoras:

$$\|v\| = \sqrt{3^2 + 4^2} = \sqrt{9 + 16} = \sqrt{25} = 5$$

La fórmula general (para cualquier número de dimensiones) es la misma idea:

$$\|v\| = \sqrt{v_1^2 + v_2^2 + \dots + v_n^2}$$

**Pasos para calcularla:** elevar cada componente al cuadrado → sumarlos → sacar la raíz cuadrada.

## Cálculo a mano con nuestros vectores

Para `a = [2, 3]`:

$$\|a\| = \sqrt{2^2 + 3^2} = \sqrt{4 + 9} = \sqrt{13} \approx 3.6056$$

Para `b = [5, 1]`:

$$\|b\| = \sqrt{5^2 + 1^2} = \sqrt{25 + 1} = \sqrt{26} \approx 5.0990$$

In [ ]:
v = np.array([3.0, 4.0])
a = np.array([2.0, 3.0])
b = np.array([5.0, 1.0])

# Forma 1: paso a paso (cuadrados -> suma -> raíz), igual que a mano
norma_v_manual = np.sqrt(np.sum(v ** 2))
print("Norma de [3,4] paso a paso:", norma_v_manual)   # esperado: 5

# Forma 2: la función de numpy
print("np.linalg.norm([3,4])     :", np.linalg.norm(v))

print()
print("Norma de a (esperado ≈ 3.6056):", np.linalg.norm(a))
print("Norma de b (esperado ≈ 5.0990):", np.linalg.norm(b))

# Un dato útil: el producto punto de un vector consigo mismo es su norma al cuadrado
print()
print("a · a        =", a @ a, " (esperado a mano: 2*2 + 3*3 = 13)")
print("norma(a) ** 2 =", np.linalg.norm(a) ** 2, " (debe ser casi igual a 13)")

**Dato útil:** $a \cdot a = \|a\|^2$. Multiplicar un vector por sí mismo con el producto punto da el cuadrado de su longitud (`13` en este caso). Esta identidad se usa para demostrar fórmulas más adelante.

La norma también funciona igual para vectores de cientos de dimensiones: son los mismos tres pasos.

---
# 7. Distancia euclidiana

## Idea intuitiva

Es la **distancia en línea recta** entre dos puntos, la que mediría una regla. **Menor distancia = más cercanos.**

## Fórmula

La distancia entre `a` y `b` es **la norma de su resta**:

$$d(a, b) = \|a - b\| = \sqrt{(a_1-b_1)^2 + (a_2-b_2)^2 + \dots + (a_n-b_n)^2}$$

Esto conecta con la sección 4: `a - b` es la flecha que va de la punta de `b` a la punta de `a`, y la **longitud de esa flecha** es justamente la distancia entre los puntos.

## Cálculo a mano con `a = [2, 3]` y `b = [5, 1]`

1. Resta: $a - b = [2-5,\; 3-1] = [-3,\; 2]$
2. Cuadrados: $(-3)^2 = 9$ y $2^2 = 4$ (el signo desaparece al elevar al cuadrado)
3. Suma: $9 + 4 = 13$
4. Raíz: $\sqrt{13} \approx 3.6056$

$$d(a, b) = \sqrt{13} \approx 3.6056$$

Como el signo desaparece, **el orden no importa**: $d(a,b) = d(b,a)$.

In [ ]:
a = np.array([2.0, 3.0])
b = np.array([5.0, 1.0])

diferencia = a - b
print("a - b          =", diferencia)                  # esperado: [-3, 2]
print("cuadrados      =", diferencia ** 2)             # esperado: [9, 4]
print("suma           =", np.sum(diferencia ** 2))     # esperado: 13
print("raíz           =", np.sqrt(np.sum(diferencia ** 2)))   # esperado: ≈ 3.6056

print()
print("np.linalg.norm(a - b) =", np.linalg.norm(a - b))
print("np.linalg.norm(b - a) =", np.linalg.norm(b - a), " (mismo resultado: el orden no importa)")

In [ ]:
distancia = np.linalg.norm(a - b)

fig, ax = preparar_plano("Distancia euclidiana entre a y b", limite_x=(-1, 8), limite_y=(-1, 6))
dibujar_flecha(ax, a, "a", "tab:blue")
dibujar_flecha(ax, b, "b", "tab:red")

# Segmento entre las puntas: su longitud ES la distancia
ax.plot([a[0], b[0]], [a[1], b[1]], linestyle="--", color="tab:green", linewidth=2)

# Etiqueta en el punto medio del segmento
punto_medio = (a + b) / 2
ax.text(punto_medio[0] + 0.1, punto_medio[1] + 0.2, f"d = {distancia:.4f}", color="tab:green", fontsize=12)

plt.show()

---
# 8. Similitud coseno

## Idea intuitiva

La distancia euclidiana mide **qué tan lejos están las puntas** de las flechas. La **similitud coseno** mide algo distinto: **el ángulo entre las flechas**, es decir, **qué tanto apuntan en la misma dirección**, sin importar qué tan largas sean.

## Fórmula

Partimos de la fórmula geométrica del producto punto (sección 5):

$$a \cdot b = \|a\|\,\|b\|\cos(\theta)$$

y despejamos $\cos(\theta)$:

$$\cos(\theta) = \frac{a \cdot b}{\|a\|\,\|b\|}$$

Es decir: **producto punto dividido entre el producto de las normas**. Al dividir entre las normas, la longitud de los vectores "desaparece" y solo queda el ángulo.

## Cómo se interpreta el resultado

El coseno siempre está entre **-1 y 1**:

| Valor | Ángulo | Significado |
|-------|--------|-------------|
| **1** | 0° | Misma dirección (máxima similitud) |
| **0** | 90° | Perpendiculares (sin relación) |
| **-1** | 180° | Direcciones opuestas |

**Mayor coseno = más parecidos** (al revés que la distancia, donde menor es más parecido). A veces se usa la **distancia coseno**, definida como $1 - \cos(\theta)$, para que "menor = más parecido" también aquí. Es importante saber cuál de las dos devuelve una herramienta; lo revisaremos cuando lleguemos a pgvector.

## Cálculo a mano con `a = [2, 3]` y `b = [5, 1]`

Ya tenemos casi todo de las secciones anteriores:

- Producto punto: $a \cdot b = 13$
- Normas: $\|a\| = \sqrt{13}$ y $\|b\| = \sqrt{26}$

Sustituimos:

$$\cos(\theta) = \frac{13}{\sqrt{13}\cdot\sqrt{26}} = \frac{13}{\sqrt{13 \cdot 26}} = \frac{13}{\sqrt{338}}$$

Como $338 = 169 \cdot 2 = 13^2 \cdot 2$, entonces $\sqrt{338} = 13\sqrt{2}$, y:

$$\cos(\theta) = \frac{13}{13\sqrt{2}} = \frac{1}{\sqrt{2}} \approx 0.7071$$

Un coseno de 0.7071 corresponde a un ángulo de **45°**. Estos vectores se eligieron así a propósito para que el resultado salga "redondo" y fácil de verificar.

In [ ]:
a = np.array([2.0, 3.0])
b = np.array([5.0, 1.0])

producto_punto = a @ b
norma_a = np.linalg.norm(a)
norma_b = np.linalg.norm(b)

coseno = producto_punto / (norma_a * norma_b)

print("a · b          =", producto_punto)               # esperado: 13
print("||a||          =", norma_a)                     # esperado: ≈ 3.6056
print("||b||          =", norma_b)                     # esperado: ≈ 5.0990
print("||a|| * ||b||  =", norma_a * norma_b)           # esperado: ≈ 18.385 (√338)
print("coseno         =", coseno)                      # esperado: ≈ 0.7071
print("1 / raíz(2)    =", 1 / np.sqrt(2), " (debe coincidir con el coseno)")

# Del coseno al ángulo: arccos es la función inversa del coseno
angulo_radianes = np.arccos(coseno)
angulo_grados = np.degrees(angulo_radianes)
print("ángulo         =", angulo_grados, "grados")     # esperado: 45

print("distancia coseno (1 - coseno) =", 1 - coseno)

In [ ]:
fig, ax = preparar_plano(f"Ángulo entre a y b: {angulo_grados:.1f}°  |  coseno = {coseno:.4f}",
                         limite_x=(-1, 8), limite_y=(-1, 6))
dibujar_flecha(ax, a, "a", "tab:blue")
dibujar_flecha(ax, b, "b", "tab:red")
plt.show()

Puedes comprobar a ojo que el ángulo entre las dos flechas se ve de aproximadamente 45°, lo que coincide con el cálculo.

---
# 9. Dos funciones reutilizables

Ya sabemos calcular las dos métricas paso a paso. Las empaquetamos en funciones para no repetir código. **Las escribimos con las piezas que ya entendimos** (norma, producto punto), sin atajos escondidos, y las comprobamos contra los resultados a mano de las secciones 7 y 8.

Una **docstring** (el texto entre triples comillas al inicio de la función) documenta qué hace.

In [ ]:
def distancia_euclidiana(a, b):
    '''Distancia en línea recta entre dos vectores. Menor = más cercanos.'''
    return np.linalg.norm(a - b)


def similitud_coseno(a, b):
    '''Coseno del ángulo entre dos vectores, entre -1 y 1. Mayor = más parecidos.'''
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))


a = np.array([2.0, 3.0])
b = np.array([5.0, 1.0])

print("distancia_euclidiana(a, b) =", distancia_euclidiana(a, b))
print("similitud_coseno(a, b)     =", similitud_coseno(a, b))

# Comprobación contra los cálculos a mano (np.isclose compara decimales con tolerancia)
print("¿Distancia ≈ raíz(13)?  ", np.isclose(distancia_euclidiana(a, b), np.sqrt(13)))
print("¿Coseno ≈ 1/raíz(2)?    ", np.isclose(similitud_coseno(a, b), 1 / np.sqrt(2)))

**¿Por qué `np.isclose` y no `==`?** Los decimales se guardan con una precisión finita, y dos cálculos distintos del "mismo" número pueden diferir en el último dígito (por ejemplo `0.30000000000000004` contra `0.3`). `np.isclose` compara con una tolerancia pequeña.

---
# 10. Cuándo usar una métrica y cuándo la otra

Las dos métricas **no siempre coinciden** al decir qué vector es "más parecido". Veamos un caso concreto.

## Analogía con documentos

Imagina que cada vector cuenta cuántas veces aparecen dos palabras, `[veces que aparece "tubo", veces que aparece "aluminio"]`, en un documento:

- Documento `a = [1, 2]`: un texto corto.
- Documento `b = [3, 6]`: un texto **largo** del mismo tema. Tiene exactamente las mismas proporciones, pero 3 veces más repeticiones. Es decir, $b = 3a$.
- Documento `c = [2, 1]`: un texto corto, pero con las proporciones invertidas (habla más de "tubo" que de "aluminio").

¿Cuál se parece más a `a`: `b` o `c`? Por tema, claramente `b`. Veamos qué dicen las métricas.

## Cálculos a mano

**Distancia euclidiana:**

- $d(a,b)$: $a-b = [-2,-4]$ → $4 + 16 = 20$ → $\sqrt{20} \approx 4.4721$
- $d(a,c)$: $a-c = [-1,\;1]$ → $1 + 1 = 2$ → $\sqrt{2} \approx 1.4142$

Según la distancia, **`c` está mucho más cerca de `a` que `b`**.

**Similitud coseno:**

- $\cos(a,b) = \dfrac{1\cdot3 + 2\cdot6}{\sqrt{5}\cdot\sqrt{45}} = \dfrac{15}{\sqrt{225}} = \dfrac{15}{15} = 1$
- $\cos(a,c) = \dfrac{1\cdot2 + 2\cdot1}{\sqrt{5}\cdot\sqrt{5}} = \dfrac{4}{5} = 0.8$

Según el coseno, **`b` es idéntico en dirección a `a`** (coseno = 1) y `c` es menos parecido.

La distancia euclidiana "castiga" que `b` sea más largo; el coseno lo ignora y solo mira hacia dónde apunta.

In [ ]:
a = np.array([1.0, 2.0])
b = np.array([3.0, 6.0])   # b = 3 * a  (misma dirección, 3 veces más largo)
c = np.array([2.0, 1.0])   # dirección distinta

print("--- Distancia euclidiana (menor = más cercano) ---")
print("d(a, b) =", distancia_euclidiana(a, b), " (esperado ≈ 4.4721)")
print("d(a, c) =", distancia_euclidiana(a, c), " (esperado ≈ 1.4142)")

print()
print("--- Similitud coseno (mayor = más parecido) ---")
print("cos(a, b) =", similitud_coseno(a, b), " (esperado 1)")
print("cos(a, c) =", similitud_coseno(a, c), " (esperado 0.8)")

In [ ]:
fig, ax = preparar_plano("b tiene la misma dirección que a; c no", limite_x=(-1, 7), limite_y=(-1, 7))
dibujar_flecha(ax, a, "a", "tab:blue")
dibujar_flecha(ax, b, "b = 3a", "tab:red")
dibujar_flecha(ax, c, "c", "tab:orange")
plt.show()

En la gráfica se ve: `a` y `b` están **sobre la misma línea** (misma dirección, ángulo 0°), mientras `c` apunta a otro lado.

**Una nota sobre los decimales:** probablemente viste `0.9999999999999999` en lugar de `1`, y `0.7999999999999998` en lugar de `0.8`. No es un error de la fórmula: los decimales se guardan en binario con precisión finita y las operaciones acumulan diferencias minúsculas (del orden de 0.0000000000000001). Por eso se usa `np.isclose` para comparar y no `==`, y por eso al mostrar resultados conviene redondear (por ejemplo `f"{valor:.4f}"`).

## ¿Entonces cuál uso?

- **Coseno** → cuando lo que importa es la **dirección** (el "tema", el significado) y no la magnitud. Es la opción típica para comparar embeddings de texto.
- **Euclidiana** → cuando la **magnitud también importa** (por ejemplo, comparar mediciones físicas donde 10 y 100 sí son muy distintos).

## El caso especial: vectores normalizados

**Normalizar** un vector es dividirlo entre su norma, para que su longitud pase a ser exactamente 1 (conserva su dirección):

$$\hat{v} = \frac{v}{\|v\|}$$

Ejemplo a mano con `v = [3, 4]` (norma 5): $\hat{v} = [3/5,\; 4/5] = [0.6,\; 0.8]$. Comprobación: $0.6^2 + 0.8^2 = 0.36 + 0.64 = 1$.

**Resultado importante:** cuando los dos vectores tienen norma 1, la distancia euclidiana y el coseno están directamente relacionados. La demostración usa lo que vimos en la sección 6 ($v\cdot v = \|v\|^2$):

$$\|a-b\|^2 = (a-b)\cdot(a-b) = a\cdot a - 2\,a\cdot b + b\cdot b = \|a\|^2 + \|b\|^2 - 2\,a\cdot b$$

Si $\|a\| = \|b\| = 1$, entonces $a\cdot b = \cos(\theta)$ y:

$$\|a-b\|^2 = 1 + 1 - 2\cos(\theta) = 2 - 2\cos(\theta)$$

Es decir, con vectores normalizados **ordenar por menor distancia euclidiana es lo mismo que ordenar por mayor coseno**. Lo comprobamos con números:

Con `u = [1, 0]` y `v = [0.6, 0.8]` (ambos de norma 1): $\cos = 1\cdot0.6 + 0\cdot0.8 = 0.6$. Distancia: $u-v = [0.4,\,-0.8]$ → $0.16 + 0.64 = 0.8$ → $d^2 = 0.8$. Y la fórmula: $2 - 2(0.6) = 0.8$. ✓

In [ ]:
def normalizar(v):
    '''Devuelve el vector con la misma dirección pero longitud 1.'''
    return v / np.linalg.norm(v)


# --- Comprobación de la relación con u = [1,0] y v = [0.6, 0.8] ---
u = np.array([1.0, 0.0])
v = np.array([0.6, 0.8])

print("norma de u:", np.linalg.norm(u), "| norma de v:", np.linalg.norm(v))   # ambas 1
print("coseno(u, v)           =", similitud_coseno(u, v))                      # esperado 0.6
print("distancia(u, v) ** 2   =", distancia_euclidiana(u, v) ** 2)             # esperado 0.8
print("2 - 2 * coseno         =", 2 - 2 * similitud_coseno(u, v))              # esperado 0.8

# --- El ejemplo de documentos, ahora con vectores normalizados ---
a_n = normalizar(a)
b_n = normalizar(b)
c_n = normalizar(c)

print()
print("Normalizados:")
print("a_n =", a_n, "| b_n =", b_n, "| c_n =", c_n)
print("d(a_n, b_n) =", distancia_euclidiana(a_n, b_n), " (b ahora sí es el más cercano)")
print("d(a_n, c_n) =", distancia_euclidiana(a_n, c_n), " (esperado a mano: raíz(2 - 2*0.8) = raíz(0.4) ≈ 0.6325)")
print("cos(a_n, c_n) =", similitud_coseno(a_n, c_n), " (el coseno no cambió: normalizar no altera ángulos)")

Al normalizar, la distancia entre `a` y `b` pasa a ser 0 (mismo punto, porque tenían la misma dirección), y `c` queda más lejos: **el orden de la distancia euclidiana ahora coincide con el del coseno**.

> **Por qué importa para lo que sigue:** según lo que conozco de forma general, muchos modelos de embeddings entregan vectores ya normalizados o se recomienda normalizarlos, pero **eso depende de cada modelo**. En la fase 2 lo comprobaremos midiendo la norma de los vectores del modelo que usemos, en lugar de suponerlo.

---
# 11. Vectores de muchas dimensiones

Todo lo anterior se hizo con 2 dimensiones para poder dibujarlo. Veamos que **las mismas funciones sirven sin cambios** con vectores de 384 dimensiones (un tamaño que mencioné como típico de algunos modelos de embeddings; el número exacto lo define cada modelo).

Para esto generamos vectores **aleatorios**. Ojo: no son embeddings reales, no representan ningún texto. Solo sirven para probar que el código escala.

- `np.random.default_rng(42)` crea un generador de números aleatorios. El `42` es la **semilla**: con la misma semilla obtienes siempre los mismos números, lo que hace reproducibles los resultados.
- `rng.normal(size=384)` genera 384 números aleatorios con distribución normal (campana de Gauss).

In [ ]:
rng = np.random.default_rng(42)

v1 = rng.normal(size=384)
v2 = rng.normal(size=384)

print("shape de v1:", v1.shape)
print("Primeros 5 números de v1:", v1[:5])   # [:5] = los primeros 5 elementos
print()
print("norma de v1             :", np.linalg.norm(v1))
print("distancia_euclidiana    :", distancia_euclidiana(v1, v2))
print("similitud_coseno        :", similitud_coseno(v1, v2))

Las mismas funciones, sin modificar, funcionaron con 384 dimensiones.

## Una curiosidad de las dimensiones altas

Dos vectores aleatorios **independientes** de muchas dimensiones suelen quedar casi perpendiculares (coseno cercano a 0). Lo comprobamos con 1000 pares de vectores y un histograma (`ax.hist` agrupa los valores en "cajas" y cuenta cuántos caen en cada una).

In [ ]:
rng = np.random.default_rng(42)

cosenos = []
for _ in range(1000):                     # "_" = variable que no vamos a usar
    x = rng.normal(size=384)
    y = rng.normal(size=384)
    cosenos.append(similitud_coseno(x, y))

cosenos = np.array(cosenos)               # lista -> array, para usar min(), max(), mean()
print("Mínimo :", cosenos.min())
print("Máximo :", cosenos.max())
print("Promedio:", cosenos.mean())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(cosenos, bins=30)                 # bins = número de cajas
ax.set_title("Coseno entre 1000 pares de vectores aleatorios (384 dimensiones)")
ax.set_xlabel("similitud coseno")
ax.set_ylabel("cantidad de pares")
ax.set_xlim(-1, 1)                        # mismo rango posible del coseno, para ver qué tan concentrados están
plt.show()

Los valores se agrupan alrededor de 0. Esto da una intuición útil para más adelante: **en un espacio de muchas dimensiones, "no relacionado" se parece a coseno ≈ 0**, y cuando dos textos reales dan un coseno claramente alto es porque el modelo los puso en direcciones parecidas.

---
# 12. Resumen

## Tabla de referencia rápida

| Concepto | Qué es | Fórmula | En numpy |
|----------|--------|---------|----------|
| **Vector** | Lista ordenada de números | $[v_1, v_2, \dots, v_n]$ | `np.array([...])` |
| **Suma / resta** | Componente con componente | $[a_1 \pm b_1, \dots]$ | `a + b`, `a - b` |
| **Escalar** | Estira o encoge, misma dirección | $[k\,a_1, \dots]$ | `k * a` |
| **Producto punto** | Un número: qué tanto apuntan al mismo lado | $\sum a_i b_i$ | `a @ b`, `np.dot(a, b)` |
| **Norma** | Longitud de la flecha | $\sqrt{\sum v_i^2}$ | `np.linalg.norm(v)` |
| **Distancia euclidiana** | Distancia en línea recta (menor = más cercano) | $\|a-b\|$ | `np.linalg.norm(a - b)` |
| **Similitud coseno** | Ángulo entre vectores (mayor = más parecido) | $\dfrac{a\cdot b}{\|a\|\,\|b\|}$ | `(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))` |
| **Distancia coseno** | Versión "menor = más parecido" | $1 - \cos\theta$ | `1 - similitud_coseno(a, b)` |
| **Normalizar** | Llevar la longitud a 1 | $v / \|v\|$ | `v / np.linalg.norm(v)` |

## Ideas clave para llevarte

1. Un vector es una lista de números; con 2 se dibuja, con 384 no, pero **las operaciones son idénticas**.
2. **Distancia euclidiana** mide qué tan separadas están las puntas. **Coseno** mide hacia dónde apuntan, ignorando la longitud.
3. Con vectores **normalizados** (longitud 1), las dos métricas ordenan igual: $\|a-b\|^2 = 2 - 2\cos\theta$.
4. Distinguir entre **similitud** (mayor = más parecido) y **distancia** (menor = más parecido) evitará errores cuando usemos bases de datos.
5. En numpy, operar con arrays "en bloque" reemplaza los ciclos.

## Lo que aprendiste de las herramientas

- **numpy:** `np.array`, `shape`, `dtype`, operaciones elemento por elemento, `np.sum`, `np.sqrt`, `@`, `np.linalg.norm`, `np.arccos`, `np.degrees`, `np.isclose`, `np.random.default_rng`.
- **matplotlib:** el patrón `fig, ax = plt.subplots(...)`, `ax.scatter`, `ax.plot`, `ax.annotate` (flechas), `ax.text`, `ax.hist`, títulos, límites, cuadrícula y `plt.show()`.

## Lo que sigue: Fase 2 — Embeddings locales

En la siguiente fase pasamos de vectores inventados a **vectores reales de texto**: qué es un modelo de embeddings, cómo descargarlo y usarlo en tu PC (sin API ni cuenta), cuántas dimensiones produce, si sus vectores vienen normalizados (lo mediremos, no lo supondremos) y cómo cambian los resultados con frases ficticias parecidas y distintas. Todo lo de esta fase se usará ahí directamente.